# Notebook04 absolute deadline watchdog definitions
Purpose: independently enforce the registered wall-clock deadline. Inputs: budget clock and explicit worker session. Outputs: watchdog evidence and cutoff status if needed. Definitions execute in under one second; the detached watcher ends when the experiment completes or reaches its original deadline. Resume: rerun Notebook04 with the same run ID; no clock extension is performed.

In [ ]:
"""Independent absolute wall-clock watchdog for the detached Notebook04 worker."""
import hashlib, json, os, shlex, signal, subprocess, sys, time
from pathlib import Path
import nbformat

def _watchdog_atomic(path,value):
    path=Path(path);temp=path.with_suffix(path.suffix+'.tmp');temp.write_text(json.dumps(value,indent=2)+'\n');temp.replace(path)

def _session_exists(session):
    return subprocess.run(['tmux','has-session','-t',session],capture_output=True).returncode==0

def _terminate_owned_session(session):
    if not _session_exists(session):return []
    pane=subprocess.check_output(['tmux','list-panes','-t',session,'-F','#{pane_pid}'],text=True)
    roots={int(v) for v in pane.split()}
    rows=[tuple(map(int,line.split())) for line in subprocess.check_output(['ps','-eo','pid=,ppid='],text=True).splitlines()]
    descendants=set(roots)
    while True:
        expanded=descendants|{pid for pid,parent in rows if parent in descendants}
        if expanded==descendants:break
        descendants=expanded
    # Only descendants of this explicitly named worker pane are terminated.
    for pid in sorted(descendants-roots,reverse=True)+sorted(roots):
        try:os.kill(pid,signal.SIGKILL)
        except ProcessLookupError:pass
    subprocess.run(['tmux','kill-session','-t',session],capture_output=True)
    return sorted(descendants)

def run_deadline_watchdog(artifact,worker_session,poll_seconds=10):
    artifact=Path(artifact);clock=json.loads((artifact/'budget_clock.json').read_text());deadline=float(clock['deadline_unix'])
    while True:
        status_path=artifact/'pilot_status.json';status=json.loads(status_path.read_text()) if status_path.exists() else {}
        if status.get('status')=='COMPLETED' or (status.get('status') in ('FAILED','INCOMPLETE') and not _session_exists(worker_session)):
            result={'status':'EXITED_TERMINAL_STATE','worker_status':status['status'],'observed_unix':time.time(),'deadline_unix':deadline}
            _watchdog_atomic(artifact/'deadline_watchdog_result.json',result);return result
        remaining=deadline-time.time()
        if remaining<=0:
            killed=_terminate_owned_session(worker_session)
            result={'status':'DEADLINE_ENFORCED','deadline_unix':deadline,'enforced_unix':time.time(),'terminated_pids':killed}
            _watchdog_atomic(status_path,{'stage':'absolute_deadline','status':'INCOMPLETE','updated_unix':time.time(),
                'reason':'Independent watchdog enforced the original three-hour wall-clock cap. Existing valid checkpoints retained.',
                'elapsed_seconds':time.time()-float(clock['started_unix'])})
            _watchdog_atomic(artifact/'deadline_watchdog_result.json',result);return result
        time.sleep(min(max(.01,float(poll_seconds)),remaining))

def launch_deadline_watchdog(root,run_id):
    root=Path(root);artifact=root/'artifacts/notebook_04'/run_id;session='fmm-deadline-'+run_id;worker='fmm-followup-'+run_id
    if _session_exists(session):return {'status':'EXISTING_SESSION','session':session}
    status_path=artifact/'pilot_status.json'
    if status_path.exists() and json.loads(status_path.read_text()).get('status')=='COMPLETED':return {'status':'COMPLETED'}
    module=root/'experiments/04_followup/deadline_watchdog.ipynb'
    code="from pathlib import Path\nimport json\nnamespace={}\npath=Path("+repr(str(module))+ ")\nfor cell in json.loads(path.read_text())['cells']:\n    if cell['cell_type']=='code' and 'ahr-export' in cell.get('metadata',{}).get('tags',[]):\n        exec(compile(''.join(cell['source']),str(path)+'#ahr-export','exec'),namespace)\nnamespace['run_deadline_watchdog'](Path("+repr(str(artifact))+"), "+repr(worker)+")\n"
    notebook=artifact/'deadline_watchdog.ipynb'
    nb=nbformat.v4.new_notebook(cells=[nbformat.v4.new_markdown_cell('# Absolute deadline watchdog\nIndependent of model execution and nested timers. Stops only the named Notebook04 worker at the original deadline. Existing atomic checkpoints remain resumable, but this run cannot extend its clock.'),nbformat.v4.new_code_cell(code)],metadata={'kernelspec':{'display_name':'Python (ahr-go)','language':'python','name':'ahr-go'}})
    nbformat.write(nb,notebook)
    command=[sys.executable,'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace','--ExecutePreprocessor.timeout=-1','--ExecutePreprocessor.kernel_name=ahr-go',str(notebook)]
    shell='exec '+shlex.join(command)+' > '+shlex.quote(str(artifact/'deadline_watchdog.log'))+' 2>&1'
    subprocess.run(['tmux','new-session','-d','-s',session,'-c',str(root),'bash','-c',shell],check=True)
    result={'status':'LAUNCHED','session':session,'worker_session':worker,'deadline_unix':json.loads((artifact/'budget_clock.json').read_text())['deadline_unix'],'module_sha256':hashlib.sha256(module.read_bytes()).hexdigest()}
    _watchdog_atomic(artifact/'deadline_watchdog_launch.json',result);return result
